# Hugging Face Audio Course — Unit 4: Build a music genre classifier

Two halves:
1. **Survey** pre-trained audio classifiers via `pipeline()`: keyword spotting, zero-shot (CLAP), language ID.
2. **Fine-tune** DistilHuBERT on GTZAN for music-genre classification, and use the result.

| # | Concept | Model |
|---|---------|-------|
| 1 | Keyword spotting | `MIT/ast-finetuned-speech-commands-v2` |
| 2 | Zero-shot classification | `laion/clap-htsat-unfused` |
| 3 | Language identification | `sanchit-gandhi/whisper-medium-fleurs-lang-id` |
| 4 | Genre classifier (fine-tuned) | `sanchit-gandhi/distilhubert-finetuned-gtzan` |

> **First run downloads several GB** of models (AST ~350 MB, CLAP ~1 GB, whisper-medium ~1.5 GB,
> genre model ~90 MB) to the Hugging Face cache. CPU only; the language-ID step is slow.
> **Fine-tuning** is impractical on CPU — see the last section for the smoke-test vs Colab/GPU split.

In [ ]:
%matplotlib inline
import librosa
import matplotlib.pyplot as plt
import IPython.display as ipd
from datasets import load_dataset, Audio
from transformers import pipeline
print("ready")

## 1. Keyword spotting
A model fine-tuned on Speech Commands to spot single command words. (We feed a cached speech clip;
since it is a full sentence, not a single word, expect `_unknown_` or low scores — the point is the pipeline.)

In [ ]:
kws = pipeline("audio-classification", model="MIT/ast-finetuned-speech-commands-v2")
clip = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]["audio"]
for p in kws({"array": clip["array"], "sampling_rate": clip["sampling_rate"]}, top_k=5):
    print(f"{p['score']:.3f}  {p['label']}")

## 2. Zero-shot audio classification (CLAP)
CLAP scores the audio against **free-text labels you choose** — no fixed class list. It expects 48 kHz.

In [ ]:
zs = pipeline("zero-shot-audio-classification", model="laion/clap-htsat-unfused")
candidate_labels = ["a trumpet playing", "a person speaking", "a dog barking", "ocean waves", "rain falling"]
y, sr = librosa.load(librosa.ex("trumpet"), sr=48_000)
preds = zs(y, candidate_labels=candidate_labels)  # zero-shot pipeline wants a raw array
for p in preds:
    print(f"{p['score']:.3f}  {p['label']}")

labels = [p["label"] for p in preds][::-1]; scores = [p["score"] for p in preds][::-1]
plt.figure(figsize=(10, 4)); plt.barh(labels, scores, color="tab:cyan")
plt.xlim(0, 1); plt.xlabel("similarity"); plt.title("CLAP zero-shot vs a trumpet clip"); plt.show()
ipd.Audio(y, rate=sr)

## 3. Language identification
Whisper-medium fine-tuned on FLEURS predicts the spoken language. (~1.5 GB; slow on CPU.)

In [ ]:
lid = pipeline("audio-classification", model="sanchit-gandhi/whisper-medium-fleurs-lang-id")
# trained on FLEURS (clean read speech): reliable on clean audio, e.g. English -> "English".
clip = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]["audio"]
for p in lid({"array": clip["array"], "sampling_rate": clip["sampling_rate"]}, top_k=5):
    print(f"{p['score']:.3f}  {p['label']}")
ipd.Audio(clip["array"], rate=clip["sampling_rate"])

## 4. The music genre classifier (the fine-tuned model)
Using the course's published DistilHuBERT fine-tuned on GTZAN. We classify a real music clip
(librosa's sweet-waltz example), which should land near `classical`.

In [ ]:
genre = pipeline("audio-classification", model="sanchit-gandhi/distilhubert-finetuned-gtzan")
clip_path = librosa.ex("sweetwaltz")
for p in genre(clip_path, top_k=5):
    print(f"{p['score']:.3f}  {p['label']}")
ipd.Audio(clip_path)

## 5. Fine-tuning DistilHuBERT on GTZAN
This is how the genre model above was made. Training takes ~1 hour on a T4 GPU and is impractical on CPU,
so the code below is **shown but gated**. To actually train:

- a quick **CPU smoke test** (proves the pipeline runs): `uv run python units/unit4_music_genre_classifier/finetune.py`
- the **full GPU/Colab run** for the hands-on (reach ≥ 87% and push to the Hub):
  `uv sync --extra training` then `UNIT4_MODE=full python .../finetune.py`

Set `RUN_TRAINING = True` below only on a GPU box.

In [ ]:
RUN_TRAINING = False  # set True on a GPU/Colab box (and `uv sync --extra training`)

In [ ]:
if RUN_TRAINING:
    import torch
    from datasets import Audio
    from transformers import (AutoFeatureExtractor, AutoModelForAudioClassification,
                              Trainer, TrainingArguments)
    import numpy as np, evaluate

    GENRES = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
    model_id = "ntu-spml/distilhubert"
    fe = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True, return_attention_mask=True)

    gtzan = load_dataset("marsyas/gtzan", trust_remote_code=True)   # ~1.2 GB; no "all" config
    split = gtzan["train"].rename_column("genre", "label").train_test_split(seed=42, test_size=0.1)
    split = split.cast_column("audio", Audio(sampling_rate=16_000))

    def preprocess(batch):
        arrays = [x["array"] for x in batch["audio"]]
        return fe(arrays, sampling_rate=16_000, max_length=16_000*30, truncation=True)
    enc = split.map(preprocess, batched=True, remove_columns=["audio", "file"])

    id2label = {str(i): g for i, g in enumerate(GENRES)}; label2id = {g: str(i) for i, g in enumerate(GENRES)}
    model = AutoModelForAudioClassification.from_pretrained(model_id, num_labels=10, id2label=id2label, label2id=label2id)
    acc = evaluate.load("accuracy")
    def compute_metrics(p): return acc.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

    args = TrainingArguments(
        "distilhubert-finetuned-gtzan",
        eval_strategy="epoch", save_strategy="epoch", learning_rate=5e-5,
        per_device_train_batch_size=8, per_device_eval_batch_size=8, num_train_epochs=10,
        warmup_steps=100, logging_steps=5, load_best_model_at_end=True,
        metric_for_best_model="accuracy", fp16=torch.cuda.is_available(), push_to_hub=True,
    )
    trainer = Trainer(model, args, train_dataset=enc["train"], eval_dataset=enc["test"],
                      processing_class=fe, compute_metrics=compute_metrics)
    trainer.train()
    trainer.push_to_hub(dataset_tags="marsyas/gtzan", dataset="GTZAN",
                        model_name="distilhubert-finetuned-gtzan",
                        finetuned_from=model_id, tasks="audio-classification")
else:
    print("skipped (set RUN_TRAINING = True on a GPU box; or run finetune.py for the CPU smoke test)")

---
### The hands-on
Fine-tune to **≥ 87%** accuracy on GTZAN (baseline 83%) and push to the Hub for your certificate. Try
more epochs, a learning-rate sweep, or a different base model (Wav2Vec2, AST). Run it on Colab/GPU.

🎉 That's Unit 4. Next: Unit 5, automatic speech recognition.